In [4]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END

# Define the state graph for the idiom module workflow
class IdiomWorkflowState(TypedDict):
    # User inputs
    amount: int
    topic: str
    level: str
    language: str

    # Generated ai's content
    idioms: List[Dict[str, Any]]
    exercises: List[Dict[str, Any]]

    # Current exercise tracking
    current_exercise_index: int
    user_answers: List[str]
    evaluation: list[Dict[str, Any]]

    # Flow control/state management
    all_exercises_completed: bool
    session_completed: bool



In [6]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

# Load the .env file
load_dotenv()
# assign key from env to langchain/openai


from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

# Make sure your API key is set in the environment
api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize model
llm = ChatOpenAI(
    model="gpt-4.1-nano",  # Make sure this model name is valid in your OpenAI account
    base_url="https://openrouter.ai/api/v1",
    temperature=0.6,
    api_key=api_key
)

In [7]:
# NODE FUNCTIONS
from pathlib import Path

def load_prompt_template(path: str) -> str:
    """Load a prompt template from a text file."""
    return Path(path).read_text()

def generate_idiom_node(state: IdiomWorkflowState):
    """Generate idioms based on user preferences"""
    # use LLM to generate idioms
    template = load_prompt_template("generate_idioms.txt")
    idioms_prompt = template.format(
        amount=state['amount'],
        topic=state['topic'],
        level=state['level'],
        language=state['language']
    )
    response = llm.invoke(idioms_prompt)
    return {
        "idioms": response,
        "curreent_exercise_index": 0
    }